# Calibration on the bundled sample data

Runs both calibrations on the small real fixtures committed under `examples/data/`
(see `examples/data/README.md`) — **no external archive needed** — and **displays the
diagnostic plots inline** (section 4).

- **CHM15k (1064 nm)** and **Mini-MPL (532 nm)** Rayleigh run fully offline.
- **CL61 / CL31 / CL51 (910 nm)** need a monthly **CAMS** file for the mandatory
  water-vapour correction (the WV LUT itself is bundled). Those cells are guarded.

## 1. Setup

In [ ]:
import tempfile
from pathlib import Path
from netCDF4 import Dataset

from calibration import calibrate_rayleigh, CalibrationOptions, InstrumentInfo
from calibration.config import InstrumentType, DataLevel
from calibration.cloud import liquid_cloud_calibration, CloudCalConfig

# locate the repo whether the notebook runs from the repo root or from examples/
ROOT = Path.cwd()
if not (ROOT / "examples" / "data").exists() and (ROOT.parent / "examples" / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"
OPTIONS = ROOT / "options.json"

CAMS = Path(CalibrationOptions.from_json(OPTIONS).cams_folder)
print("sample data :", DATA)
print("CAMS present:", CAMS.exists(), "->", CAMS, "(needed only for 910 nm)")

def l2(wmo, date):
    return DATA / "L2" / wmo / "2026" / date[4:6] / f"L2_{wmo}_A{date}.nc"

def info_from(wmo, itype, ncpath):
    with Dataset(ncpath) as ds:
        lat = float(ds.variables["station_latitude"][...])
        lon = float(ds.variables["station_longitude"][...])
        alt = float(ds.variables["station_altitude"][...])
    return InstrumentInfo(site_name=wmo, wmo_id=wmo, identifier="A",
                          instrument_type=itype, latitude=lat, longitude=lon, altitude=alt)

def rayleigh_options(plots=False, out=None):
    o = CalibrationOptions.from_json(OPTIONS)
    o.data_level = DataLevel.L2_DAILY
    o.folder_root = DATA / "L2"
    o.cams_folder = CAMS
    o.abs_cs_lookup_table = Path("")          # use the bundled WV LUT
    o.folder_output = Path(out) if out else Path(tempfile.mkdtemp())
    o.plot_main = bool(plots)
    if hasattr(o, "plot_all"):
        o.plot_all = False
    return o

## 2. Rayleigh calibration — CHM15k (1064 nm), self-contained

A clear night at Payerne. 1064 nm is insensitive to water vapour, so this runs offline.

In [ ]:
res = calibrate_rayleigh("20260225", info_from("0-20000-0-06610", InstrumentType.CHM15k,
                         l2("0-20000-0-06610", "20260225")), rayleigh_options())
print("CHM15k  flag =", res.flag, "(", res.flag_meaning, ")   lidar constant =", f"{res.lidar_constant:.3e}")

### Mini-MPL (532 nm), self-contained — a clear night at Lille

In [ ]:
res = calibrate_rayleigh("20260423", info_from("0-20000-0-07014", InstrumentType.MINI_MPL,
                         l2("0-20000-0-07014", "20260423")), rayleigh_options())
print("Mini-MPL  flag =", res.flag, "   lidar constant =", f"{res.lidar_constant:.3e}")

## 3. CL61 (910 nm) — Rayleigh **and** cloud

Both apply the water-vapour correction (bundled LUT + monthly CAMS); the cells run only
if CAMS is present.

In [ ]:
if not CAMS.exists():
    print("CAMS not found - skipping the 910 nm CL61 examples. Set cams_folder in options.json.")
else:
    res = calibrate_rayleigh("20260304", info_from("0-756-4-EERLCL61", InstrumentType.CL61,
                             l2("0-756-4-EERLCL61", "20260304")), rayleigh_options())
    print("CL61 Rayleigh  flag =", res.flag, "   lidar constant =", f"{res.lidar_constant:.3e}")

    cloud = liquid_cloud_calibration(CloudCalConfig(
        nc_file=str(l2("0-756-4-EERLCL61", "20260414")),
        cams_folder=str(CAMS), apply_wv_correction=1, aerosol_lidar_ratio=50))  # bundled WV LUT
    print("CL61 cloud     in-cloud profiles =", cloud.n_profiles, "   cal_median =", cloud.cal_median)

### Cloud calibration that yields a constant — CL31 / CL51 (overcast)

In [ ]:
if CAMS.exists():
    for wmo, date in [("0-20000-0-06602", "20260220"), ("0-20000-0-02998", "20260116")]:
        r = liquid_cloud_calibration(CloudCalConfig(nc_file=str(l2(wmo, date)),
                cams_folder=str(CAMS), apply_wv_correction=1, aerosol_lidar_ratio=50))
        print(f"{wmo}  n={r.n_profiles:4d}  calibration coefficient C = {r.cal_median:.3f}")
else:
    print("CAMS not found - skipping the 910 nm cloud examples.")

## 4. Show the diagnostic plots inline

With `plot_main=True` the calibration writes diagnostic PNGs (RCS time-series, annotated
RCS, and the compact dashboard). Here we run CHM15k with plotting on and display them in
the notebook.

In [ ]:
from IPython.display import Image, display

o = rayleigh_options(plots=True)                       # plot_main on, temp output dir
calibrate_rayleigh("20260225", info_from("0-20000-0-06610", InstrumentType.CHM15k,
                   l2("0-20000-0-06610", "20260225")), o)

pdir = o.folder_output / "plots" / "0-20000-0-06610" / "2026"
pngs = sorted(pdir.glob("*.png"))
print(f"{len(pngs)} diagnostic plots in {pdir}")
for png in pngs:
    print("
" + png.name)
    display(Image(filename=str(png)))

The wide **compact dashboard** is the last image: molecular-fit panels, window-search
heatmaps (slope / intercept / R² with the chosen window marked), the sensitivity grid,
and the annotated RCS pcolor (molecular layer, cloud detections, used vs flagged/unused
profiles).

See `tests/test_sample_data.py` for these fixtures as automated tests and
`examples/data/README.md` for the full list.